# 08.6a 预训练音频生成模型总览

本 Notebook 统一检查 MusicGen、AudioLDM2、Stable Audio Open、YuE 和 ACE-Step 的 runner 状态，写出状态表，帮助后续模型 Notebook 只运行当前环境可用的部分。


## 运行环境索引

第八章的大模型不要全部装进同一个虚拟环境。按 Notebook 选择 kernel：

| Notebook | 模型 | kernel / 环境 |
|:---|:---|:---|
| `08_6b` | MusicGen | `Python 3.11 (chapter08-audiocraft)` |
| `08_6c` | AudioLDM2 | `Python 3.11 (chapter08-diffusers)` |
| `08_6d` | Stable Audio Open | `Python 3.10 (chapter08-stable-audio)` |
| `08_6e` | YuE | `Python 3.10 (chapter08-yue)` / `CODE/venv_ch08_yue` 或官方 CUDA 环境 |
| `08_6f` | ACE-Step | `Python 3.10 (chapter08-acestep)` 或官方 CUDA 环境 |

具体安装、下载、登录命令写在对应 Notebook 的开头。本页只汇总当前环境是否已经可运行。


In [ ]:
from pathlib import Path
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from _common.paths import portable_path
from model_runners.ace_step import ACEStepRunner
from model_runners.audioldm2 import AudioLDM2Runner
from model_runners.base import write_runner_status_table
from model_runners.conditioning import write_conditioning_table
from model_runners.musicgen import MusicGenRunner
from model_runners.stable_audio_open import StableAudioOpenRunner
from model_runners.yue import YuERunner
from runners.run_pretrained_available import REQUEST_PLAN_FIELDS, build_request_plan_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)


In [ ]:
runners = [
    MusicGenRunner(),
    AudioLDM2Runner(),
    StableAudioOpenRunner(),
    YuERunner(),
    ACEStepRunner(),
]
statuses = [runner.check_environment() for runner in runners]
status_path = OUTPUT_TABLES / "08_model_runner_status.csv"
write_runner_status_table(status_path, statuses)
df = pd.DataFrame([status.as_row() for status in statuses])
display(df)


In [ ]:
summary = (
    df.groupby("status", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("status")
)
display(summary)
print("status table:", rel(status_path))


In [ ]:
request_plan = build_request_plan_rows(statuses, notebook_root=ROOT)
request_plan_path = OUTPUT_TABLES / "08_pretrained_request_plan.csv"
pd.DataFrame(request_plan).to_csv(request_plan_path, index=False, columns=REQUEST_PLAN_FIELDS)
display(pd.DataFrame(request_plan))
print("request plan:", rel(request_plan_path))


In [ ]:
next_actions = df[["model_name", "status", "next_action"]].copy()
display(next_actions)


## 这些模型到底根据什么生成？

下表把当前 Notebook 实际传给模型的条件拆开。文本到音频不是“无条件采样”：prompt、歌词、genre 文件、负面 prompt、duration、seed、音频前缀或 LoRA 训练集都会进入模型或生成脚本。


In [ ]:
conditioning_path = OUTPUT_TABLES / "08_pretrained_conditioning_matrix.csv"
conditioning_rows = write_conditioning_table(conditioning_path)
display(pd.DataFrame(conditioning_rows))
print("conditioning matrix:", rel(conditioning_path))
